# Prototype A: data centers and water stress
Step 1 of the map, kept small on purpose: every data center in the database, over county water stress.
More layers (price change, restrictions, planned capacity) get added one at a time once this reads well.

- **Dots:** every facility with usable coordinates, all statuses (operating, planned, stalled, cancelled). Hover for name, city, status and MW.
- **Shading:** share of the county's area in High or Extremely High baseline water stress (WRI Aqueduct 4.0, one 2023 snapshot, not a trend).
- Water stress shows where water is scarce, not whether data centers caused it.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("data", "jppolizzi") else Path.cwd()
sys.path[:0] = [str(ROOT / ".pylibs"), str(ROOT / "scripts")]
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import proto_data as pdx
CLEAN = ROOT / "data" / "clean"
import plotly.graph_objects as go, geopandas as gpd, json

c = pdx.county_frame()
geo = gpd.read_file(CLEAN / "counties_simplified.geojson")
d = pd.read_csv(CLEAN / "datacenters.csv")
d = d[d.coord_ok]
print(len(d), "data centers plotted;", len(c), "counties shaded")

1733 data centers plotted; 3144 counties shaded


In [2]:
fig = go.Figure()
fig.add_trace(go.Choropleth(geojson=json.loads(geo.to_json()), locations=c["geoid"], featureidkey="properties.geoid", z=c["share_high_or_extreme"] * 100,
    colorscale=[[0, "#f7fbff"], [1, "#2171b5"]], zmin=0, zmax=100, marker_line_width=0, showscale=True,
    colorbar=dict(title="% of county in<br>high/extreme<br>water stress", len=0.5),
    text=c["county_name"] + ", " + c["state"], hovertemplate="%{text}<br>%{z:.0f}% high/extreme stress<extra></extra>"))
fig.add_trace(go.Scattergeo(lon=d["lon"], lat=d["lat"], mode="markers", name="data center (any status)",
    marker=dict(size=5, color="#d94801", opacity=0.75, line=dict(width=0)),
    text=d["facility_name"].fillna("(unnamed)") + "<br>" + d["city"].fillna("") + ", " + d["state"] + " | " + d["status"].fillna("status unknown") +
         "<br>" + d["mw"].map(lambda v: "MW not reported" if pd.isna(v) else f"{v:,.0f} MW"), hovertemplate="%{text}<extra></extra>"))
fig.update_layout(height=620, margin=dict(l=0, r=0, t=50, b=0), showlegend=True, legend=dict(x=0.02, y=0.08, bgcolor="rgba(255,255,255,0.8)"),
    title=dict(text="Data center locations over county water stress (WRI Aqueduct 4.0 baseline)", x=0.02, font=dict(size=16)),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showland=False, showlakes=False, showframe=False))
fig.write_html(ROOT / "jppolizzi" / "prototype_a_map.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()

Saved as `jppolizzi/prototype_a_map.html`. Facilities without coordinates are skipped. Next layer to add (one at a time): planned capacity, then restrictions.

## Grid strain: balancing authority regions
The grid is run by **balancing authorities** (BAs), which balance supply and demand hour by hour (PJM, MISO, ERCOT, ...). EIA-930 reports hourly demand for each, so this is the unit for grid strain.

- **Regions** are approximate: built from the counties each BA's utilities serve (EIA-861, `scripts/build_ba_regions.py`). A county served by several BAs goes to the one whose utility has the most residential customers, so small BAs mostly disappear into their neighbours.
- **Shading:** growth in annual peak demand, 2019 to 2025 (EIA-930). Grey = no complete 2019 and 2025 data.
- **Dots:** operating (blue) and planned (orange) data centers.
- Peak growth also reflects weather, population and electrification, so this map does not separate the data center share.

In [3]:
g = pdx.ba_grid_table().set_index("ba")
regions = gpd.read_file(CLEAN / "ba_regions.geojson").merge(g[["peak_base_mw", "peak_end_mw", "peak_growth_pct", "dc_operating"]], left_on="ba_code", right_index=True, how="left")
regions["dc_operating"] = regions["dc_operating"].fillna(0)
regions["label"] = (regions["ba_name"].fillna(regions["ba_code"]) + " (" + regions["ba_code"] + ")<br>" +
    regions.apply(lambda r: "no complete 2019/2025 data" if pd.isna(r.peak_growth_pct) else
                  f"peak {r.peak_base_mw / 1000:,.1f} GW (2019) to {r.peak_end_mw / 1000:,.1f} GW (2025), {r.peak_growth_pct:+.1f}%", axis=1) +
    "<br>operating data centers in served counties: " + regions["dc_operating"].astype(int).astype(str))
has = regions[regions.peak_growth_pct.notna()]; none = regions[regions.peak_growth_pct.isna()]

fig = go.Figure()
fig.add_trace(go.Choropleth(geojson=json.loads(none.to_json()), locations=none["ba_code"], featureidkey="properties.ba_code", z=[0] * len(none),
    colorscale=[[0, "#e0e0e0"], [1, "#e0e0e0"]], showscale=False, marker_line_color="white", marker_line_width=0.8, text=none["label"], hovertemplate="%{text}<extra></extra>"))
fig.add_trace(go.Choropleth(geojson=json.loads(has.to_json()), locations=has["ba_code"], featureidkey="properties.ba_code", z=has["peak_growth_pct"],
    colorscale=[[0, "#fcd9c8"], [0.5, "#ef6548"], [1, "#7f0000"]], zmin=0, zmax=20, marker_line_color="white", marker_line_width=0.8,
    colorbar=dict(title="peak demand<br>growth 2019-25 (%)", len=0.5, ticksuffix="%", tickvals=[0, 5, 10, 15, 20], ticktext=["0% or less", "5%", "10%", "15%", "20%+"]), text=has["label"], hovertemplate="%{text}<extra></extra>"))
d2 = d[d.status_group.isin(["operating", "pipeline"])]
for grp, color, name in (("pipeline", "#e8833a", "planned data center"), ("operating", "#1f5fa8", "operating data center")):
    s_ = d2[d2.status_group == grp]
    fig.add_trace(go.Scattergeo(lon=s_["lon"], lat=s_["lat"], mode="markers", name=name, marker=dict(size=4, color=color, opacity=0.8, line=dict(width=0)),
        text=s_["facility_name"].fillna("(unnamed)") + "<br>" + s_["city"].fillna("") + ", " + s_["state"], hovertemplate="%{text}<extra></extra>"))
big = regions.nlargest(10, "n_counties").copy()
pt = big.to_crs("EPSG:5070").geometry.representative_point().to_crs("EPSG:4326")
fig.add_trace(go.Scattergeo(lon=pt.x, lat=pt.y, mode="text", text=big["ba_code"], textfont=dict(size=12, color="#111", family="Arial Black"), hoverinfo="skip", showlegend=False))
fig.update_layout(height=620, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.02, y=0.08, bgcolor="rgba(255,255,255,0.8)"),
    title=dict(text="Grid regions (balancing authorities): peak demand growth 2019-2025 and data center locations", x=0.02, font=dict(size=16)),
    geo=dict(scope="usa", projection=dict(type="albers usa"), showland=True, landcolor="#e6e6e6", showlakes=False, showframe=False,
             showsubunits=True, subunitcolor="#ffffff", subunitwidth=0.4))
fig.add_annotation(text="grey = no region assigned or no complete 2019/2025 data", x=0.02, y=0.0, xref="paper", yref="paper", showarrow=False, font=dict(size=11, color="#555"), xanchor="left")
fig.write_html(ROOT / "jppolizzi" / "prototype_a_grid_map.html", include_plotlyjs=True, config={"displayModeBar": False})
fig.show()
display(g.sort_values("dc_operating", ascending=False).head(10)[["peak_base_mw", "peak_end_mw", "peak_growth_pct", "dc_operating"]].round(1))

,peak_base_mw,peak_end_mw,peak_growth_pct,dc_operating
ba,,,,
PJM,152315.0,160560.0,5.4,294.0
SOCO,45515.0,46490.0,2.1,88.0
ERCO,74533.0,83597.0,12.2,71.0
MISO,116600.0,120343.0,3.2,37.0
SWPP,50511.0,54411.0,7.7,28.0
BPAT,10275.0,11537.0,12.3,28.0
WALC,1919.0,1995.0,4.0,23.0
TVA,29568.0,35086.0,18.7,17.0
NYIS,30397.0,31857.0,4.8,17.0


Saved as `jppolizzi/prototype_a_grid_map.html`. Next: test whether BAs with more data centers grew faster (Prototype B), and add PJM capacity prices.